In [ ]:
from pyhealth.datasets import MIMIC4Dataset, get_dataloader, split_by_sample, split_by_patient
from pyhealth.tasks import Readmission30DaysMIMIC4, MortalityPredictionMIMIC4
from pyhealth.trainer import Trainer
from pyhealth.models import Transformer
from models.mamba import Mamba

MIMIC4_PATH = "datasets/mimic-iv-2.2"

In [ ]:
dataset = MIMIC4Dataset(
    ehr_root=MIMIC4_PATH,
    ehr_tables=["patients", "admissions", "diagnoses_icd", "procedures_icd", "prescriptions"],
    dev=False
)

In [ ]:
from tasks.tasks import BinaryLengthOfStayPredictionMIMIC4, MortalityPrediction31DaysMIMIC4

TASK = "readmission_30_days"

if TASK == "readmission_30_days":
    dataset_with_task = dataset.set_task(
        task=Readmission30DaysMIMIC4(),
        cache_dir="cache/readmission_prediction",
        num_workers=4,
    )
elif TASK == "length_of_stay_prediction":
    dataset_with_task = dataset.set_task(
        task=BinaryLengthOfStayPredictionMIMIC4(),
        cache_dir="cache/length_of_stay_prediction",
        num_workers=1
    )
elif TASK == "mortality_prediction":
    dataset_with_task = dataset.set_task(
        task=MortalityPrediction31DaysMIMIC4(),
        cache_dir="cache/mortality_prediction",
        num_workers=1,
    )

if TASK == "mortality_prediction":
    train_dataset, val_dataset, test_dataset = split_by_patient(dataset_with_task, ratios=[0.765, 0.085, 0.15])
else:
    train_dataset, val_dataset, test_dataset = split_by_sample(dataset_with_task, ratios=[0.765, 0.085, 0.15])

In [ ]:
train_loader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_loader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_loader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
from models.mamba_mpy import Mamba as MambaMPY
from models.mamba2_mpy import Mamba2 as Mamba2MPY
from models.mamba_mpy_add import MambaAdd
from models.jamba_mpy import Jamba
import torch

# model = Mamba(dataset=dataset_with_task, embedding_dim=128, num_layers=16, dropout=0.1)
# model = Mamba(dataset=dataset_with_task, embedding_dim=128, num_layers=2, dropout=0.1)
# model = MambaMPY(dataset=dataset_with_task, embedding_dim=128, num_layers=16, dropout=0.1)
# model = Mamba2MPY(dataset=dataset_with_task, embedding_dim=128, num_layers=2, dropout=0.1)
# model = Jamba(dataset=dataset_with_task, embedding_dim=128, num_layers=16, dropout=0.1)
model = MambaAdd(dataset=dataset_with_task, embedding_dim=128, num_layers=16, dropout=0.1)
# model = Transformer(dataset=dataset_with_task, embedding_dim=128, num_layers=16, dropout=0.1)
trainer = Trainer(model=model, metrics=["roc_auc", "pr_auc", "f1"], device="cuda")


In [ ]:
import torch
from utils.optimization import build_linear_scheduler_with_warmup_and_decay

num_epochs = 20
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=num_epochs,
    monitor="roc_auc",
    optimizer_class=torch.optim.AdamW,
    optimizer_params={"lr": 5e-5},
    scheduler_class_or_builder=build_linear_scheduler_with_warmup_and_decay,
    scheduler_params={"n_steps": len(train_loader) * num_epochs, "warmup_ratio": 0.1, "decay_ratio": 0.9},
)


In [ ]:
metrics = trainer.evaluate(test_loader)

In [ ]:
metrics